# 02 — The Topological Clock

**Persistent homology of the patent citation network in sliding time windows.**

This is the novel core of the project. We compute Betti numbers (β₀, β₁, β₂)
and persistence entropy for CPC section-pair subgraphs across time, looking for
topological signatures — loops, voids, structural complexity — that might
precede technological breakthroughs.

**β₀** = connected components (fragmentation of knowledge)  
**β₁** = 1-dimensional loops (circular knowledge flows between fields)  
**β₂** = 2-dimensional voids (enclosed cavities — rare but structurally meaningful)

---

In [1]:
# %% Imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from src.utils import DATA_DIR, get_logger, log_memory
from src.topology import sliding_window_topology, compute_persistence, betti_numbers, persistence_entropy
from src.graph import cpc_subgraph_nx
from src.plotting import set_style, save_figure, PALETTE, PALETTE_LIST, CPC_COLORS, CPC_LABELS, year_axis, confidence_band

set_style()
logger = get_logger('nb02')

In [2]:
# %% Load data
patents = pd.read_parquet(DATA_DIR / 'patents.parquet')
citations = pd.read_parquet(DATA_DIR / 'citations.parquet')
cpc_map = pd.read_parquet(DATA_DIR / 'cpc_map.parquet')

print(f'Patents: {len(patents):,}  |  Citations: {len(citations):,}  |  CPC: {len(cpc_map):,}')

Patents: 8,451,545  |  Citations: 118,011,718  |  CPC: 17,668,819


## Define CPC Section Pairs

We select pairs that span different technological domains where breakthroughs
are known to have occurred at the intersection.

In [ ]:
# %% CPC section pairs for analysis
# With A100 (167GB RAM, 12 CPUs) we can analyze ALL 28 section pairs
CPC_SECTIONS = list('ABCDEFGH')
SECTION_PAIRS = []
for i, a in enumerate(CPC_SECTIONS):
    for b in CPC_SECTIONS[i + 1:]:
        SECTION_PAIRS.append((a, b))

# Human-readable descriptions for key pairs
PAIR_DESCRIPTIONS = {
    ('A', 'C'): 'Human Necessities × Chemistry (biotech/pharma)',
    ('G', 'H'): 'Physics × Electricity (semiconductor/computing)',
    ('B', 'G'): 'Operations × Physics (manufacturing tech)',
    ('A', 'G'): 'Human Necessities × Physics (medical devices)',
    ('C', 'H'): 'Chemistry × Electricity (batteries/energy)',
    ('B', 'C'): 'Operations × Chemistry (materials processing)',
    ('F', 'H'): 'Mech. Engineering × Electricity (power systems)',
    ('G', 'C'): 'Physics × Chemistry (analytical instruments)',
}

print(f'Analyzing ALL {len(SECTION_PAIRS)} CPC section pairs (A100 environment):')
for a, b in SECTION_PAIRS:
    desc = PAIR_DESCRIPTIONS.get((a, b), '')
    suffix = f'  — {desc}' if desc else ''
    print(f'  ({a}, {b}){suffix}')

In [ ]:
# %% Compute sliding-window topology for each pair
# This is the most expensive computation in the project.
# Results are cached to data/topology_cache/ after first run.
# A100 environment: 30K nodes (ripser limit), 4-hop distances, ALL 28 CPC pairs

topology_results = {}

for sec_a, sec_b in SECTION_PAIRS:
    pair_key = f'{sec_a}_{sec_b}'
    desc = PAIR_DESCRIPTIONS.get((sec_a, sec_b), '')
    print(f'\n=== ({sec_a}, {sec_b}){f": {desc}" if desc else ""} ===')
    
    result = sliding_window_topology(
        citations=citations,
        cpc_map=cpc_map,
        section_a=sec_a,
        section_b=sec_b,
        window_years=5,
        stride_years=1,
        start_year=1985,
        end_year=2023,
        max_dim=2,
        max_nodes=30_000,
    )
    
    topology_results[pair_key] = result
    print(f'  Windows computed: {len(result)}')
    if len(result) > 0:
        print(f'  β₁ range: {result["beta_1"].min()} - {result["beta_1"].max()}')
        print(f'  PE range: {result["persistence_entropy"].min():.3f} - {result["persistence_entropy"].max():.3f}')

log_memory('After all topology computations')

## Figure 1: β₁ Time Series for Each CPC Pair

β₁ counts 1-dimensional loops in the citation network between two CPC sections.
A rising β₁ indicates increasing circular knowledge flow — patents forming closed
citation cycles across disciplinary boundaries.

In [ ]:
# %% Figure 1: β₁ time series
nrows, ncols = 4, 7  # 28 pairs in a 4×7 grid
fig, axes = plt.subplots(nrows, ncols, figsize=(28, 16), sharex=True)
axes = axes.flatten()

for i, (sec_a, sec_b) in enumerate(SECTION_PAIRS):
    pair_key = f'{sec_a}_{sec_b}'
    ax = axes[i]
    
    if pair_key in topology_results and len(topology_results[pair_key]) > 0:
        df = topology_results[pair_key]
        color = PALETTE_LIST[i % len(PALETTE_LIST)]
        ax.plot(df['year'], df['beta_1'], color=color, linewidth=1.5)
        ax.fill_between(df['year'], 0, df['beta_1'], alpha=0.15, color=color)
    
    ax.set_title(f'({sec_a}, {sec_b})', fontsize=10)
    if i % ncols == 0:
        ax.set_ylabel('β₁')
    year_axis(ax, 1985, 2023)

fig.suptitle('Figure 1: β₁ (1-Dimensional Loops) Across All 28 CPC Section Pairs Over Time', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, '02_beta1_time_series')

## Figure 2: Persistence Entropy Time Series

Persistence entropy measures the diversity of topological feature lifetimes.
Higher entropy = more complex topological structure.

In [ ]:
# %% Figure 2: Persistence entropy
nrows, ncols = 4, 7
fig, axes = plt.subplots(nrows, ncols, figsize=(28, 16), sharex=True)
axes = axes.flatten()

for i, (sec_a, sec_b) in enumerate(SECTION_PAIRS):
    pair_key = f'{sec_a}_{sec_b}'
    ax = axes[i]
    
    if pair_key in topology_results and len(topology_results[pair_key]) > 0:
        df = topology_results[pair_key]
        color = PALETTE_LIST[i % len(PALETTE_LIST)]
        ax.plot(df['year'], df['persistence_entropy'], color=color, linewidth=1.5)
    
    ax.set_title(f'({sec_a}, {sec_b})', fontsize=10)
    if i % ncols == 0:
        ax.set_ylabel('Persistence Entropy')
    year_axis(ax, 1985, 2023)

fig.suptitle('Figure 2: Persistence Entropy Across All 28 CPC Section Pairs Over Time', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, '02_persistence_entropy')

## Figure 3: Persistence Diagram Gallery

Representative persistence diagrams showing the birth-death structure
of topological features for a selected CPC pair at different time points.

In [ ]:
# %% Figure 3: Persistence diagram gallery
# Select one CPC pair and show diagrams at different time points
GALLERY_PAIR = ('A', 'C')  # biotech/pharma
GALLERY_YEARS = [1990, 2000, 2010, 2020]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, yr in zip(axes, GALLERY_YEARS):
    # Build the subgraph for this window
    win_start = pd.Timestamp(f'{yr - 4}-01-01')
    win_end = pd.Timestamp(f'{yr}-12-31')
    mask = (pd.to_datetime(citations['citing_date']) >= win_start) & (pd.to_datetime(citations['citing_date']) <= win_end)
    window_cites = citations[mask]
    
    G = cpc_subgraph_nx(window_cites, cpc_map, GALLERY_PAIR[0], GALLERY_PAIR[1], max_nodes=30_000)
    
    from src.topology import reduce_graph, compute_persistence
    G = reduce_graph(G, max_nodes=30_000)
    result = compute_persistence(G, max_dim=1)
    
    # Plot persistence diagram
    for dim, color, label in [(0, PALETTE['blue'], 'H₀'), (1, PALETTE['red'], 'H₁')]:
        if dim < len(result['dgms']) and len(result['dgms'][dim]) > 0:
            dgm = result['dgms'][dim]
            finite = dgm[np.isfinite(dgm[:, 1])]
            if len(finite) > 0:
                ax.scatter(finite[:, 0], finite[:, 1], s=10, alpha=0.5, c=color, label=label)
    
    # Diagonal line
    lims = ax.get_xlim()
    ax.plot([0, max(lims[1], 5)], [0, max(lims[1], 5)], 'k--', alpha=0.3)
    ax.set_title(f'{yr}')
    ax.set_xlabel('Birth')
    ax.set_ylabel('Death')
    ax.legend(fontsize=8)

fig.suptitle(f'Figure 3: Persistence Diagrams for ({GALLERY_PAIR[0]}, {GALLERY_PAIR[1]}) at Four Time Points', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, '02_persistence_diagrams')

## Figure 4: β₁ vs. CPC Mixing Rate

Does the topological signal (β₁) track the simple metric (cross-section citation rate),
or does topology reveal structure that mixing rate misses?

In [ ]:
# %% Figure 4: β₁ vs mixing rate comparison
from src.metrics import cpc_mixing_rate

mixing = cpc_mixing_rate(citations, cpc_map)
mixing = mixing[(mixing['year'] >= 1985) & (mixing['year'] <= 2023)]

# Average β₁ across all pairs per year
all_beta1 = []
for pair_key, df in topology_results.items():
    if len(df) > 0:
        all_beta1.append(df[['year', 'beta_1']])

if all_beta1:
    combined = pd.concat(all_beta1)
    avg_beta1 = combined.groupby('year')['beta_1'].mean().reset_index()

    fig, ax1 = plt.subplots(figsize=(12, 6))

    ax1.plot(mixing['year'], mixing['mixing_rate'], color=PALETTE['blue'], linewidth=2, label='CPC Mixing Rate')
    ax1.set_ylabel('Mixing Rate', color=PALETTE['blue'])
    ax1.set_xlabel('Year')

    ax2 = ax1.twinx()
    ax2.plot(avg_beta1['year'], avg_beta1['beta_1'], color=PALETTE['red'], linewidth=2, label='Mean β₁ (all pairs)')
    ax2.set_ylabel('Mean β₁', color=PALETTE['red'])

    ax1.set_title('Figure 4: CPC Mixing Rate vs. Topological Signal (β₁)')
    year_axis(ax1, 1985, 2023)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    fig.tight_layout()
    save_figure(fig, '02_beta1_vs_mixing')

## Figure 5: β₁ Heatmap Across All CPC Pairs

Where is topology changing fastest? Which section pairs show the most
topological activity over time?

In [ ]:
# %% Figure 5: β₁ heatmap
# Build a matrix: CPC pairs × years
pair_labels = [f'({a},{b})' for a, b in SECTION_PAIRS]
all_years = sorted(set().union(*[set(df['year']) for df in topology_results.values() if len(df) > 0]))

heatmap_data = np.full((len(SECTION_PAIRS), len(all_years)), np.nan)

for i, (sec_a, sec_b) in enumerate(SECTION_PAIRS):
    pair_key = f'{sec_a}_{sec_b}'
    if pair_key in topology_results and len(topology_results[pair_key]) > 0:
        df = topology_results[pair_key]
        for _, row in df.iterrows():
            if row['year'] in all_years:
                j = all_years.index(row['year'])
                heatmap_data[i, j] = row['beta_1']

fig, ax = plt.subplots(figsize=(20, 10))
im = ax.imshow(heatmap_data, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax.set_yticks(range(len(pair_labels)))
ax.set_yticklabels(pair_labels, fontsize=8)

# Show every 5th year on x-axis
tick_positions = [i for i, y in enumerate(all_years) if y % 5 == 0]
ax.set_xticks(tick_positions)
ax.set_xticklabels([all_years[i] for i in tick_positions])
ax.set_xlabel('Year')

plt.colorbar(im, ax=ax, label='β₁', shrink=0.8)
ax.set_title('Figure 5: β₁ Across All 28 CPC Section Pairs Over Time')
fig.tight_layout()
save_figure(fig, '02_beta1_heatmap')

## Summary

Key observations from the Topological Clock:

1. **β₁ trends** — Do we see 1-dimensional loops forming between CPC sections over time?
2. **Persistence entropy** — Is topological complexity increasing?
3. **Topology vs. mixing rate** — Do they track each other, or does topology reveal hidden structure?
4. **Which pairs are most active?** — The heatmap identifies the topological hotspots.

Notebook 03 will define the breakthrough ground truth, and Notebook 04 will test
whether these topological signals systematically precede breakthroughs.